# Step 2 — Test: resampler.py

验证 1min → 15min 聚合结果是否正确。

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from src.data.loader import load_raw_ohlcv
from src.data.resampler import resample_ohlcv

In [ ]:
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

df_1min  = load_raw_ohlcv(cfg)
df_15min = resample_ohlcv(df_1min, cfg)

In [ ]:
# 基本信息
print('1min  shape:', df_1min.shape)
print('15min shape:', df_15min.shape)
print('Any NaN?', df_15min.isna().any().any())
df_15min.head()

In [ ]:
# 画 15min 收盘价（前 2000 根，约 20 天）
sample = df_15min['Close'].iloc[:2000]
plt.figure(figsize=(16, 4))
plt.plot(sample.index, sample.values)
plt.title('NQ Close Price — 15min bars (first 2000 bars)')
plt.xlabel('Time')
plt.ylabel('Price')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# 验证第一根 15min bar 的聚合是否正确
first_start = df_15min.index[0]
first_end   = first_start + pd.Timedelta('15min')
src = df_1min.loc[first_start:first_end].iloc[:-1]

bar = df_15min.iloc[0]
print('=== 第一根 15min bar 验证 ===')
print(f"Open  : {bar['Open']}  == 1min 第一根 Open  {src['Open'].iloc[0]}  → {'✅' if bar['Open']  == src['Open'].iloc[0]  else '❌'}")
print(f"High  : {bar['High']}  == 1min max High    {src['High'].max()}   → {'✅' if bar['High']  == src['High'].max()   else '❌'}")
print(f"Low   : {bar['Low']}   == 1min min Low     {src['Low'].min()}    → {'✅' if bar['Low']   == src['Low'].min()    else '❌'}")
print(f"Close : {bar['Close']} == 1min 最后 Close  {src['Close'].iloc[-1]} → {'✅' if bar['Close'] == src['Close'].iloc[-1] else '❌'}")
print(f"Volume: {bar['Volume']} == 1min sum Volume {src['Volume'].sum()} → {'✅' if bar['Volume'] == src['Volume'].sum()  else '❌'}")